In [30]:
from openai import OpenAI
import json
import requests

In [31]:
id = "1xwf351iu329qg"

In [32]:
runpod_url = f"https://{id}-8000.proxy.runpod.net"

In [33]:
health_url = f"{runpod_url}/health"

In [34]:
health_check_res = requests.get(health_url)
if health_check_res.status_code != 200:
    print("헬스 체크 실패")
    exit()

print(health_check_res.json())

{'status': 'ok'}


In [35]:
model_list_url = f"{runpod_url}/v1/models"

In [36]:
model_list_res = requests.get(model_list_url)

if model_list_res.status_code != 200:
    print("모델 목록 조회 실패")
    exit()

model_list = model_list_res.json()

print(model_list)

model_name = model_list['data'][0]['id']

{'object': 'list', 'data': [{'id': 'google/gemma-4-E4B-it', 'object': 'model', 'created': 1776221815, 'owned_by': 'vllm', 'root': 'google/gemma-4-E4B-it', 'parent': None, 'max_model_len': 32768, 'permission': [{'id': 'modelperm-b74da5a646765c17', 'object': 'model_permission', 'created': 1776221815, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [37]:
chat_url = f"{runpod_url}/v1/chat/completions"

In [38]:
import base64

def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [53]:
def build_messages(image_path: str, text: str) -> list:
    ext = image_path.rsplit(".", 1)[-1].lower()
    mime_map = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "webp": "image/webp", "gif": "image/gif"}
    mime_type = mime_map.get(ext, "image/jpeg")

    b64 = encode_image(image_path)

    print(f"{b64[:100]}...")

    return [{
        "role": "user",
        "content": [
            {"type": "text", "text": text},
            {"type": "image_url", "image_url": {"url": f"data:{mime_type};base64,{b64}"}}
        ]
    }]



In [51]:
image_path = "D:\TEST\orchestrator_loop\image_data\\flat_bill.jpg"

In [56]:
messages = build_messages(image_path, "영수증 내용을 알려줘")

/9j/4AAQSkZJRgABAAEAAQABAAD//gAfTEVBRCBUZWNobm9sb2dpZXMgSW5jLiBWMS4wMQD/2wCEAAkGBwgHBgkIBwgKCgkLDhgP...


In [ ]:
payload = {
    "model" : model_name,
    "messages": messages,
    "max_tokens": 4000,
    "stream": False
}

In [58]:
res = requests.post(chat_url, json=payload)

In [59]:
print(res.json())


{'id': 'chatcmpl-8a51a5ab44dd8a5c', 'object': 'chat.completion', 'created': 1776222528, 'model': 'google/gemma-4-E4B-it', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '제공해주신 영수증 내용은 다음과 같습니다. (텍스트로 변환 및 정리했습니다.)\n\n---\n\n**[영수증 정보]**\n\n* **상호명:** 농협 (농협 로고 포함되어 있는 것으로 보입니다.)\n* **사업자 정보:** (일부 내용이 잘려있거나 가려져 있어 전체적인 주소/사업자 번호 확인 어려움)\n* **연락처:** 031-******\n\n**[구매 내역]**\n\n| 품목명 (코드) | 수량 | 단가 | 금액 |\n|---|---|---|---|\n| 001 P 곡모뇌우유 (900ML) | 1 | 1,350 | 1,350 |\n| 002 P 양파 | 1 | 3,300 | 3,300 |\n| 003 P 우유 | 1 | 500 | 500 |\n| 004 P 액밀 | 1 | 750 | 750 |\n| 005 P 하선정 바로먹기 좋은 장아찌 (150g) | 1 | 1,380 | 1,380 |\n| 006 P 브로콜리 | 1 | 1,280 | 1,280 |\n\n**[결제 정보]**\n\n* **합계 금액:** 8,560원\n* **VAT (부가세):** 718원\n* **최종 결제 금액:** 9,275원 (계산된 값으로 추정)\n\n**[거래 상세]**\n\n* **거래 일시:** 2015-11-03 16:31:53\n* **거래 번호:** 0002-00085\n* **포인트:**\n    * 적립 금액: 40점\n    * 사용 가능 포인트: 14.198점\n    * 사용 포인트: 14.190점\n\n**[기타 메모]**\n* *"...농협에서 가능"*\n* *"...조은 하루 되세요!"*\n\n---\n\n**요약:

In [ ]:
payload['stream'] = True

res = requests.post(chat_url, json=payload)

is_reasoning = False

for line in res.iter_lines():
    if line:
        decoded = line.decode('utf-8')

        if decoded.startswith('data: '):
            data = decoded.split('data: ')[1]

            if data == '[DONE]':
                break;
            
            json_data = json.loads(data)
            
            tokens = json_data['choices'][0]['delta']

            if tokens.get('reasoning', None):
                if not is_reasoning: 
                    print("<REASONING>")
                    is_reasoning = True
                print(tokens['reasoning'], end='', flush=True)
            
            if tokens.get('content', None):
                if is_reasoning:
                    print("\n<RESPONSE>")
                    is_reasoning = False
                print(tokens['content'], end='', flush=True)

data: {"id":"chatcmpl-ab3ee961afd3a071","object":"chat.completion.chunk","created":1776222664,"model":"google/gemma-4-E4B-it","choices":[{"index":0,"delta":{"role":"assistant","content":""},"logprobs":null,"finish_reason":null}],"prompt_token_ids":null}
data: {"id":"chatcmpl-ab3ee961afd3a071","object":"chat.completion.chunk","created":1776222664,"model":"google/gemma-4-E4B-it","choices":[{"index":0,"delta":{"content":"제"},"logprobs":null,"finish_reason":null,"token_ids":null}]}
data: {"id":"chatcmpl-ab3ee961afd3a071","object":"chat.completion.chunk","created":1776222664,"model":"google/gemma-4-E4B-it","choices":[{"index":0,"delta":{"content":"공"},"logprobs":null,"finish_reason":null,"token_ids":null}]}
data: {"id":"chatcmpl-ab3ee961afd3a071","object":"chat.completion.chunk","created":1776222664,"model":"google/gemma-4-E4B-it","choices":[{"index":0,"delta":{"content":"해주"},"logprobs":null,"finish_reason":null,"token_ids":null}]}
data: {"id":"chatcmpl-ab3ee961afd3a071","object":"chat.com